# Multi-Observation Grism Fitting Demo

This notebook demonstrates how to use geko's `run_geko_fit_multi()` function to jointly fit multiple grism observations.

## Overview

The multi-observation framework allows you to:
- Fit multiple R-dispersion observations at different position angles
- (Future) Fit R and C dispersion observations jointly
- Share galaxy parameters across observations while computing separate likelihoods

## Key Components

1. **run_geko_fit_multi()**: High-level wrapper function that handles everything
2. **observations_config**: List of dictionaries specifying each observation's parameters

This demo uses the same data twice to demonstrate the framework. In practice, you would use different observations at different position angles.

In [ ]:
# Import necessary packages
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import numpyro

from geko.fitting import run_geko_fit_multi

# Configure JAX and Numpyro
jax.config.update('jax_enable_x64', True)
if 'gpu' in str(jax.devices()):
    print('Using GPU')
    numpyro.set_platform('gpu')
else:
    print('Using CPU')
    
numpyro.set_host_device_count(2)
numpyro.enable_validation()

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Step 1: Set up Parameters

In [ ]:
# Set up paths
geko_path = '/Users/lola/geko/'  # Change this to your geko folder path

# Define parameters
source_id = 191250
field = 'manual'
output_name = 'my_galaxy'
master_catalog = geko_path + 'demo/simple_fit_demo_files/catalogs/my_galaxies_cat'
emission_line = 'H_alpha'
parametric = True
save_runs_path = geko_path + 'demo/simple_fit_demo_files/'

# Manual field parameters
manual_psf_name = 'webbPSF_F444W.fits'
manual_pysersic_file = 'summary_191250_image_F150W_svi.cat'
grism_file = 'spec_2d_FRESCO_F444W_ID191250_comb.fits'

# Processing parameters
grism_filter = 'F444W'
delta_wave_cutoff = 0.02
factor = 1  # Set low for demo speed (typically 5)
wave_factor = 1  # Set low for demo speed (typically 9)
model_name = 'Disk'

# MCMC parameters
num_chains = 2
num_warmup = 100  # Reduced for demo (use 500+ for real analysis)
num_samples = 100  # Reduced for demo (use 500+ for real analysis)

# Configure observations
# For this demo, we use the same observation twice with the same parameters
# In practice, you would specify different grism files at different position angles
observations_config = [
    {
        'grism_file': grism_file,
        'theta_rot': 0.0,  # rotation angle in degrees
        'dispersion': 'R',  # R (row) or C (column)
        'name': 'obs1'
    },
    {
        'grism_file': grism_file,  # Same file for demo
        'theta_rot': 0.0,  # Same angle for demo
        'dispersion': 'R',
        'name': 'obs2'
    }
]

print("Parameters set successfully")
print(f"Data directory: {save_runs_path}")
print(f"Source ID: {source_id}")
print(f"Observations to fit: {len(observations_config)}")

## Step 2: Run Multi-Observation Fitting

The `run_geko_fit_multi()` function handles:
1. Loading and preprocessing data for each observation
2. Creating GrismObservation objects
3. Setting up priors (from PySersic or config)
4. Running MCMC inference with shared parameters
5. Post-processing results for each observation

In [ ]:
print("Running multi-observation fitting...")
print("This may take a few minutes...")
print()

# Run the multi-observation fitting
inf_data, results = run_geko_fit_multi(
    observations_config=observations_config,
    output=output_name,
    master_cat=master_catalog,
    line=emission_line,
    parametric=parametric,
    save_runs_path=save_runs_path,
    num_chains=num_chains,
    num_warmup=num_warmup,
    num_samples=num_samples,
    source_id=source_id,
    field=field,
    grism_filter=grism_filter,
    delta_wave_cutoff=delta_wave_cutoff,
    factor=factor,
    wave_factor=wave_factor,
    model_name=model_name,
    manual_psf_name=manual_psf_name,
    manual_pysersic_file=manual_pysersic_file
)

print("\nFitting complete!")
print(f"Results available for: {list(results.keys())}")

## Step 3: Analyze Posterior Results

In [ ]:
# Print posterior statistics
import arviz as az

posterior = inf_data.posterior
params = ['PA', 'i', 'Va', 'r_t', 'sigma0', 'r_eff', 'n']

print("Posterior Statistics (shared across all observations)")
print("="*80)
print(f"{'Parameter':<15} {'Median':<12} {'16%':<12} {'84%':<12} {'Mean':<12} {'Std':<12}")
print("-"*80)

for param in params:
    if param in posterior:
        samples = np.concatenate(posterior[param][:])
        median = np.percentile(samples, 50)
        p16 = np.percentile(samples, 16)
        p84 = np.percentile(samples, 84)
        mean = np.mean(samples)
        std = np.std(samples)
        print(f"{param:<15} {median:<12.3f} {p16:<12.3f} {p84:<12.3f} {mean:<12.3f} {std:<12.3f}")

## Step 4: Visualize Results for Each Observation

In [ ]:
# Plot results for each observation
for obs_name, obs_results in results.items():
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Results for {obs_name}', fontsize=16, fontweight='bold')
    
    # Model prediction (median)
    im0 = axs[0].imshow(obs_results['model_map'], origin='lower', cmap='PuBu', aspect='auto')
    axs[0].set_title('Model Prediction (median)')
    axs[0].set_xlabel('Wavelength [pixels]')
    axs[0].set_ylabel('Spatial [pixels]')
    plt.colorbar(im0, ax=axs[0], label='Flux')
    
    # Velocity field
    im1 = axs[1].imshow(obs_results['model_velocities_low'], origin='lower', cmap='RdBu_r', aspect='equal', vmin=-150, vmax=150)
    axs[1].set_title('Velocity Field (km/s)')
    axs[1].set_xlabel('X [pixels]')
    axs[1].set_ylabel('Y [pixels]')
    plt.colorbar(im1, ax=axs[1], label='v (km/s)')
    
    # Flux map
    im2 = axs[2].imshow(obs_results['fluxes_mean'], origin='lower', cmap='inferno', aspect='equal')
    axs[2].set_title('Flux Map')
    axs[2].set_xlabel('X [pixels]')
    axs[2].set_ylabel('Y [pixels]')
    plt.colorbar(im2, ax=axs[2], label='Flux')
    
    plt.tight_layout()
    plt.show()

## Step 5: MCMC Diagnostics

In [ ]:
# Trace plots for key parameters
print("MCMC Trace Plots")
print("="*60)
az.plot_trace(inf_data, var_names=['Va', 'sigma0', 'PA', 'i'])
plt.tight_layout()
plt.show()

# Summary statistics
print("\nMCMC Summary Statistics")
print("="*60)
print(az.summary(inf_data, var_names=['PA', 'i', 'Va', 'r_t', 'sigma0', 'r_eff', 'n']))

In [ ]:
## Summary

This notebook demonstrated the simplified multi-observation fitting workflow using `run_geko_fit_multi()`:

1. ✓ Set up observation configurations (grism file, rotation angle, dispersion)
2. ✓ Call `run_geko_fit_multi()` with all parameters
3. ✓ Analyze posterior statistics for shared parameters
4. ✓ Visualize model predictions for each observation
5. ✓ Check MCMC diagnostics

## Key Advantages of `run_geko_fit_multi()`

- **Simple API**: Single function call handles all preprocessing and fitting
- **Shared parameters**: Galaxy parameters sampled once across all observations
- **Automatic handling**: Creates GrismObservation objects internally
- **Clean results**: Returns inference data and model predictions in organized format

## Comparison with Single-Observation Fitting

**Single-observation** (`run_geko_fit()`):
- Fits one observation at a time
- Simpler for single position angle analysis

**Multi-observation** (`run_geko_fit_multi()`):
- Jointly fits multiple observations
- Better constraints from multiple viewing angles
- Shares parameters across observations
- Useful for multi-PA or R+C dispersion data

## Next Steps

- Apply to real multi-angle observations at different `theta_rot` values
- Use longer chains for convergence (500+ samples, 500+ warmup)
- Compare results with single-observation fits
- (Future) Test with mixed R and C dispersion observations

In [ ]:
# Run the multi-observation inference
fit.run_inference_multi(
    observations=observations,
    masks=None,  # Auto-generate masks
    num_samples=100,  # Reduced for demo (use 500+ for real analysis)
    num_warmup=100,
    num_chains=2,
    step_size=0.1,
    adapt_step_size=True,
    target_accept_prob=0.8
)

print("\nMCMC sampling completed!")
print(f"Posterior shape: {fit.inference_data.posterior.dims}")

## Step 7: Post-Process Results

The `compute_model_parametric_multi()` method generates model predictions for each observation.

In [ ]:
# Post-process results
print("Post-processing results for each observation...")
results = kin_model.compute_model_parametric_multi(
    fit.inference_data,
    observations
)

print(f"\nResults computed for {len(results)} observations")
print(f"Observation names: {list(results.keys())}")
print(f"\nResult keys for each observation:")
for key in results[observations[0].name].keys():
    print(f"  - {key}")

## Step 8: Analyze Posterior Results

Check posterior statistics for the shared parameters.

In [ ]:
# Print posterior statistics
import arviz as az

posterior = fit.inference_data.posterior
params = ['PA', 'i', 'Va', 'r_t', 'sigma0', 'r_eff', 'n']

print("Posterior Statistics (shared across both observations)")
print("="*80)
print(f"{'Parameter':<15} {'Median':<12} {'16%':<12} {'84%':<12} {'Mean':<12} {'Std':<12}")
print("-"*80)

for param in params:
    if param in posterior:
        samples = np.concatenate(posterior[param][:])
        median = np.percentile(samples, 50)
        p16 = np.percentile(samples, 16)
        p84 = np.percentile(samples, 84)
        mean = np.mean(samples)
        std = np.std(samples)
        print(f"{param:<15} {median:<12.3f} {p16:<12.3f} {p84:<12.3f} {mean:<12.3f} {std:<12.3f}")

## Step 9: Visualize Results

Plot model predictions vs observations for each observation.

In [ ]:
# Plot results for each observation
for obs_name, obs_results in results.items():
    obs = next(o for o in observations if o.name == obs_name)
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Results for {obs_name}', fontsize=16, fontweight='bold')
    
    # Observation
    im0 = axs[0].imshow(obs.obs_map, origin='lower', cmap='PuBu', aspect='auto')
    axs[0].set_title('Observed Grism Spectrum')
    axs[0].set_xlabel('Wavelength [pixels]')
    axs[0].set_ylabel('Spatial [pixels]')
    plt.colorbar(im0, ax=axs[0], label='Flux')
    
    # Model
    im1 = axs[1].imshow(obs_results['model_map'], origin='lower', cmap='PuBu', aspect='auto')
    axs[1].set_title('Model Prediction (median)')
    axs[1].set_xlabel('Wavelength [pixels]')
    axs[1].set_ylabel('Spatial [pixels]')
    plt.colorbar(im1, ax=axs[1], label='Flux')
    
    # Residuals
    residuals = (obs.obs_map - obs_results['model_map']) / obs.obs_error
    im2 = axs[2].imshow(residuals, origin='lower', cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
    axs[2].set_title('Residuals (σ)')
    axs[2].set_xlabel('Wavelength [pixels]')
    axs[2].set_ylabel('Spatial [pixels]')
    plt.colorbar(im2, ax=axs[2], label='σ')
    
    plt.tight_layout()
    plt.show()

## Step 10: MCMC Diagnostics

In [ ]:
# Trace plots for key parameters
print("MCMC Trace Plots")
print("="*60)
az.plot_trace(fit.inference_data, var_names=['Va', 'sigma0', 'PA', 'i'])
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\nMCMC Summary Statistics")
print("="*60)
print(az.summary(fit.inference_data, var_names=['PA', 'i', 'Va', 'r_t', 'sigma0', 'r_eff', 'n']))

## Summary

This notebook demonstrated:

1. ✓ Loading real JWST grism data using the preprocessing pipeline
2. ✓ Creating `GrismObservation` objects to bundle observation data
3. ✓ Running multi-observation MCMC with `run_inference_multi()`
4. ✓ Post-processing with `compute_model_parametric_multi()`
5. ✓ Shared parameters sampled once across observations
6. ✓ Separate model predictions and likelihoods for each observation

## Key Differences from Single-Observation Fitting

**Single-observation** (`simple_fit_demo.ipynb`):
- Uses `run_geko_fit()` wrapper function
- Fits one observation at a time

**Multi-observation** (this notebook):
- Uses preprocessing to load data manually
- Creates `GrismObservation` objects for each observation
- Uses `fit.run_inference_multi()` for joint fitting
- Uses `compute_model_parametric_multi()` for post-processing

## Next Steps

- Test with observations at different `theta_rot` angles
- (Future) Test with R and C dispersion observations jointly
- Use longer chains for convergence validation (500+ samples)
- Apply to your own multi-angle JWST observations